# 📝 [제 13편] 오차역전파 (2) - 은닉층 공식과 MNIST 고속 학습

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「오차역전파 (III) 은닉층」, 「(IV) 일반 공식」, 「(V) 오차역전파를 이용한 MNIST 인식」 (ML_LEC_25, 26, 27)

---

## 🎯 학습 목표
1. 은닉층 가중치의 오차역전파 공식 loss_2 = (loss_3 · W3ᵀ) × A2(1 − A2) 를 연쇄법칙으로 설명할 수 있다.
2. 층이 여러 개인 신경망에 쓸 수 있는 오차역전파 일반 공식을 말할 수 있다.
3. 오차역전파로 MNIST 신경망을 학습시키고, 수치미분과 속도·정확도를 비교할 수 있다.

---

## 🧭 지난 시간 복습
* 제 12편 출력층 공식:  loss_3 = (A3 − T) × A3(1 − A3),  ∂E/∂W3 = A2ᵀ · loss_3,  ∂E/∂b3 = loss_3

[⬅ 이전 : 제 12편 오차역전파 (1) - 연쇄법칙으로 한꺼번에 미분하기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/12_backprop_1.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/00_index.ipynb)　|　[다음 : 제 14편 CNN - 합성곱으로 사진의 특징 찾기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/14_cnn.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

In [ ]:
#@title ▶ 수치미분, sigmoid 함수 (실행만 하세요)
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x; fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x; fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)
        x[idx] = tmp_val
        it.iternext()
    return grad

def sigmoid(z):
    return 1 / (1 + np.exp(-z))
print('준비 완료')

## 📖 개념 다지기 1. 은닉층 가중치는 길이 두 갈래!

2 - 2 - 2 신경망에서 은닉층 가중치 w<sub>11</sub><sup>(2)</sup> (입력 노드 1 → 은닉 노드 1) 를 생각합니다.

w<sub>11</sub><sup>(2)</sup> 가 바뀌면 → a<sub>1</sub><sup>(2)</sup> 가 바뀌고 → 출력 노드 **1 과 2 모두** 가 바뀝니다. 그래서 연쇄법칙의 길도 두 갈래입니다.

> ∂E/∂w<sub>11</sub><sup>(2)</sup> = { **(출력 노드 1 의 손실) × w<sub>11</sub><sup>(3)</sup>** + **(출력 노드 2 의 손실) × w<sub>21</sub><sup>(3)</sup>** } × a<sub>1</sub><sup>(2)</sup>(1 − a<sub>1</sub><sup>(2)</sup>) × a<sub>1</sub><sup>(1)</sup>

* { } 안 : 출력층의 손실들이 가중치를 타고 **거꾸로** 은닉 노드 1 로 모여든 것
* 여기에 은닉 노드의 sigmoid 미분 a(1 − a) 를 곱한 것을 **은닉층 손실 (loss_2)** 이라 합니다.

### 행렬로 쓰면

> **loss_2 = (loss_3 · W3<sup>T</sup>) × A2 × (1 − A2)**
> **∂E/∂W2 = A1<sup>T</sup> · loss_2,　 ∂E/∂b2 = loss_2**

💡 출력층 공식과 **모양이 똑같습니다!**  (이전 층 출력)ᵀ · (현재 층 손실)

### [연습 1] 은닉층 손실 손으로 계산하기

loss_3 = (0.1, −0.2),　W3 = ( 1　2 ) / ( 3　4 ) (은닉 2 × 출력 2),　A2 = (0.5, 0.5) 일 때

① loss_3 · W3<sup>T</sup> = (0.1 × 1 + (−0.2) × 2,　0.1 × 3 + (−0.2) × 4) = ( 　　,　　 )
② A2 × (1 − A2) = (0.25, 0.25)
③ loss_2 = ① × ② = ( 　　　,　　　 )

In [ ]:
loss_3 = np.array([[0.1, -0.2]])
W3 = np.array([[1, 2], [3, 4]])
A2 = np.array([[0.5, 0.5]])
loss_2 = np.dot(loss_3, W3.T) * A2 * (1 - A2)
print('loss_3 · W3ᵀ =', np.dot(loss_3, W3.T))
print('loss_2       =', loss_2)

## 💻 실습 1. 은닉층 공식 검증 (수치미분과 비교)

In [ ]:
A1 = np.array([[0.5, 0.8]]); T = np.array([[1.0, 0.0]])
np.random.seed(3)
W2 = np.random.rand(2, 2); b2 = np.random.rand(2)
W3 = np.random.rand(2, 2); b3 = np.random.rand(2)

def loss():
    A2 = sigmoid(A1 @ W2 + b2); A3 = sigmoid(A2 @ W3 + b3)
    return 0.5 * np.sum((T - A3)**2)

# 피드포워드
A2 = sigmoid(A1 @ W2 + b2)
A3 = sigmoid(A2 @ W3 + b3)
# 오차역전파 : 출력층 → 은닉층 순서로!
loss_3 = (A3 - T) * A3 * (1 - A3)
loss_2 = (loss_3 @ W3.T) * A2 * (1 - A2)
print('[오차역전파] ∂E/∂W2 =\n', A1.T @ loss_2, '\n ∂E/∂b2 =', loss_2[0])

f = lambda x : loss()
print('\n[수치미분]   ∂E/∂W2 =\n', numerical_derivative(f, W2), '\n ∂E/∂b2 =', numerical_derivative(f, b2))

> **[확인 1]** 은닉층 공식도 수치미분과 ( 같은 / 다른 ) 결과를 낸다.

## 📖 개념 다지기 2. 오차역전파 일반 공식 (은닉층이 여러 개여도 OK)

**현재 층 / 이전 층 / 다음 층** 으로 이름을 붙이면 모든 층이 같은 공식을 따릅니다.

| 무엇 | 공식 |
|:---|:---|
| 출력층 손실 | (출력층 출력 − 정답) × 출력층 출력 × (1 − 출력층 출력) |
| 은닉층(현재 층) 손실 | (다음 층 손실 · 다음 층 W<sup>T</sup>) × 현재 층 출력 × (1 − 현재 층 출력) |
| 현재 층 W 의 미분 | (이전 층 출력)<sup>T</sup> · 현재 층 손실 |
| 현재 층 b 의 미분 | 현재 층 손실 |

입력층 → 은닉층1 → 은닉층2 → 출력층 (W2, W3, W4) 인 신경망이라면

　loss_4 = (A4 − T) × A4(1 − A4) 　→　 loss_3 = (loss_4 · W4<sup>T</sup>) × A3(1 − A3) 　→　 loss_2 = (loss_3 · W3<sup>T</sup>) × A2(1 − A2)

　W4 ← W4 − α × (A3<sup>T</sup> · loss_4),　W3 ← W3 − α × (A2<sup>T</sup> · loss_3),　W2 ← W2 − α × (A1<sup>T</sup> · loss_2)

> **[연습 2]** 은닉층이 3개인 신경망 (W2, W3, W4, W5) 에서 loss_3 을 구하려면 ( 　 ) 층의 손실과 W( 　 ) 가 필요하다.

## 🎬 애니메이션 1. 오차가 거꾸로 퍼져 나가는 모습

입력 2 - 은닉 4 - 은닉 3 - 출력 2 신경망입니다. **순전파**(파랑)로 출력을 계산한 뒤, 출력층의 **오차(빨강)** 가 가중치를 타고 한 층씩 거꾸로 전달됩니다. 원의 색이 진할수록 그 노드의 손실(|loss|)이 큽니다.

In [ ]:
#@title 🎬 애니메이션 1 실행 (오차의 역전파)
np.random.seed(7)
sizes = [2, 4, 3, 2]
Ws = [np.random.randn(sizes[i], sizes[i+1]) for i in range(3)]
bs = [np.random.rand(sizes[i+1]) for i in range(3)]
x = np.array([[0.9, 0.1]]); tt = np.array([[0.0, 1.0]])
As = [x]
for W_, b_ in zip(Ws, bs):
    As.append(sigmoid(As[-1] @ W_ + b_))
losses = [None]*4
losses[3] = (As[3] - tt) * As[3] * (1 - As[3])
for l in [2, 1]:
    losses[l] = (losses[l+1] @ Ws[l].T) * As[l] * (1 - As[l])
pos = [[(i*3, (sizes[i]-1)/2 - j) for j in range(sizes[i])] for i in range(4)]
vmax = max(np.abs(L).max() for L in losses[1:])
frames = [('f', l) for l in range(4) for _ in range(5)] + [('b', l) for l in [3, 2, 1] for _ in range(7)] + [('b', 1)]*8
names = ['입력층', '은닉층 1', '은닉층 2', '출력층']
fig, ax = plt.subplots(figsize=(11, 5))
def draw(f):
    mode, k = frames[f]
    ax.clear(); ax.set_xlim(-1, 11.5); ax.set_ylim(-2.6, 2.6); ax.axis('off')
    for l in range(3):
        for i, p in enumerate(pos[l]):
            for j, q in enumerate(pos[l+1]):
                if mode == 'f':
                    on = l < k; col = 'royalblue'
                else:
                    on = l >= k; col = 'crimson'          # 손실이 지나간 가중치
                w = Ws[l][i, j]
                ax.plot([p[0], q[0]], [p[1], q[1]], color=col if on else 'lightgray', lw=0.8 + 1.5*abs(w) if on else 0.8, alpha=0.8 if on else 0.4)
    for l in range(4):
        ax.text(l*3, 2.35, names[l], ha='center', fontsize=12)
        for j, (cx, cy) in enumerate(pos[l]):
            if mode == 'b' and l >= k and l >= 1:
                v = abs(losses[l][0, j]); fc = plt.cm.Reds(0.15 + 0.85*v/vmax); txt = f'{losses[l][0, j]:+.3f}'
            elif mode == 'f' and l <= k or mode == 'b':
                fc = plt.cm.Blues(0.15 + 0.6*As[l][0, j]); txt = f'{As[l][0, j]:.2f}'
            else:
                fc, txt = 'white', ''
            ax.add_patch(plt.Circle((cx, cy), 0.38, fc=fc, ec='k', lw=1.5, zorder=3))
            ax.text(cx, cy, txt, ha='center', va='center', fontsize=9, zorder=4)
    if mode == 'b' or k == 3:
        ax.text(10.2, 0.5, f'정답 T\n{tt[0].tolist()}', fontsize=11, va='center')
    ax.set_title('① 순전파 : 출력 a 계산 (원 안 = 출력값)' if mode == 'f' else '② 역전파 : 손실(loss)이 출력층 → 은닉층2 → 은닉층1 로 전달 (원 안 = loss)', fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=170)
plt.close(fig)
show_anim(anim)

## 💻 실습 2. 오차역전파 MNIST 신경망 (원본 강의 코드)

원본 강의(ML_LEC_27)의 NeuralNetwork 클래스입니다. 달라진 점:
* 가중치 초기화: `np.random.randn(...) / np.sqrt(입력 노드 수 / 2)` (**He 초기화** - 학습이 잘 되도록 처음 값을 알맞은 크기로)
* `train` 에서 수치미분 대신 **오차역전파 공식** 사용
* 학습률 0.3

In [ ]:
#@title ▶ MNIST 불러오기 (제 11편과 같음)
try:
    from tensorflow.keras.datasets import mnist
    (x_tr, t_tr), (x_te, t_te) = mnist.load_data()
    training_data = np.c_[t_tr, x_tr.reshape(-1, 784)].astype(np.float64)
    test_data     = np.c_[t_te, x_te.reshape(-1, 784)].astype(np.float64)
    print('✅ MNIST 불러오기 성공')
except Exception as e:
    from sklearn.datasets import load_digits
    d = load_digits()
    data = np.c_[d.target, d.images.reshape(-1, 64) * (255 / 16)]
    training_data, test_data = data[:1500], data[1500:]
    print('⚠️ MNIST 를 불러오지 못해 8 × 8 손글씨 데이터로 대신합니다:', e)
IMG = int(np.sqrt(training_data.shape[1] - 1))
print('training_data.shape =', training_data.shape, ', test_data.shape =', test_data.shape)

In [ ]:
from datetime import datetime

class NeuralNetwork:
    def __init__(self, input_nodes, hidden_nodes, output_nodes, learning_rate):
        self.input_nodes, self.hidden_nodes, self.output_nodes = input_nodes, hidden_nodes, output_nodes
        # 은닉층 가중치 W2 (784 × 100), 바이어스 b2
        self.W2 = np.random.randn(self.input_nodes, self.hidden_nodes) / np.sqrt(self.input_nodes / 2)
        self.b2 = np.random.rand(self.hidden_nodes)
        # 출력층 가중치 W3 (100 × 10), 바이어스 b3
        self.W3 = np.random.randn(self.hidden_nodes, self.output_nodes) / np.sqrt(self.hidden_nodes / 2)
        self.b3 = np.random.rand(self.output_nodes)
        self.learning_rate = learning_rate

    def feed_forward(self):
        delta = 1e-7
        self.Z1 = self.input_data;  self.A1 = self.input_data              # 입력층
        self.Z2 = np.dot(self.A1, self.W2) + self.b2; self.A2 = sigmoid(self.Z2)   # 은닉층
        self.Z3 = np.dot(self.A2, self.W3) + self.b3; self.A3 = sigmoid(self.Z3)   # 출력층
        return -np.sum(self.target_data*np.log(self.A3 + delta) + (1 - self.target_data)*np.log((1 - self.A3) + delta))

    def loss_val(self):
        return self.feed_forward()

    def train(self, input_data, target_data):        # input_data (1 × 784), target_data (1 × 10)
        self.target_data = target_data
        self.input_data = input_data
        loss_val = self.feed_forward()                 # ① 피드포워드
        # ② 출력층 손실 → W3, b3 갱신
        loss_3 = (self.A3 - self.target_data) * self.A3 * (1 - self.A3)
        # ③ 은닉층 손실 (W3 를 갱신하기 '전' 값으로 계산)
        loss_2 = np.dot(loss_3, self.W3.T) * self.A2 * (1 - self.A2)
        self.W3 = self.W3 - self.learning_rate * np.dot(self.A2.T, loss_3)
        self.b3 = self.b3 - self.learning_rate * loss_3
        self.W2 = self.W2 - self.learning_rate * np.dot(self.A1.T, loss_2)
        self.b2 = self.b2 - self.learning_rate * loss_2
        return loss_val

    def predict(self, input_data):
        A2 = sigmoid(np.dot(input_data, self.W2) + self.b2)
        A3 = sigmoid(np.dot(A2, self.W3) + self.b3)
        return np.argmax(A3)

    def accuracy(self, test_data):
        matched = 0
        for row in test_data:
            data = (row[1:] / 255.0 * 0.99) + 0.01
            if self.predict(data) == int(row[0]):
                matched += 1
        return matched / len(test_data)
print('NeuralNetwork (오차역전파 버전) 준비 완료')

> ⚠️ 원본 코드에서는 W3 를 먼저 갱신한 뒤 loss_2 를 계산하지만, 수학적으로 정확하려면 **갱신 전의 W3** 로 loss_2 를 구해야 합니다. 이 학습지에서는 이 부분을 바로잡았습니다.

### 속도 비교 : 사진 1장 학습 시간

In [ ]:
import time
np.random.seed(0)
nn = NeuralNetwork(IMG*IMG, 100, 10, 0.3)
row = training_data[0]
x1 = ((row[1:] / 255.0) * 0.99 + 0.01).reshape(1, -1)
t1 = np.zeros((1, 10)) + 0.01; t1[0, int(row[0])] = 0.99

start = time.time(); nn.train(x1, t1); t_bp = time.time() - start

nn.target_data, nn.input_data = t1, x1
f = lambda x : nn.feed_forward()
start = time.time(); numerical_derivative(f, nn.W3); t_nd3 = time.time() - start   # W3 만 수치미분
t_nd = t_nd3 * (nn.W2.size + nn.b2.size + nn.W3.size + nn.b3.size) / nn.W3.size     # 전체로 환산

print(f'오차역전파 : 사진 1장 {t_bp*1000:.2f} ms')
print(f'수치미분   : 사진 1장 약 {t_nd:.2f} 초 (W3 만 재고 전체 개수로 환산)')
print(f'→ 오차역전파가 약 {t_nd/t_bp:,.0f} 배 빠르다!')

> **[확인 2]** 오차역전파는 수치미분보다 약 ( 　　　 ) 배 빠르다.

### 학습 (학습 데이터 전체 1번 = 1 epoch, 30초 ~ 1분)

In [ ]:
np.random.seed(0)
nn = NeuralNetwork(IMG*IMG, 100, 10, 0.3)
epochs = 1 if len(training_data) > 10000 else 15
snapshots, acc_hist = [], []                        # 애니메이션용 기록
check = max(len(training_data)*epochs // 30, 1)
start_time = datetime.now()
count = 0
for ep in range(epochs):
    for step, row in enumerate(training_data):
        target_data = np.zeros(10) + 0.01
        target_data[int(row[0])] = 0.99
        input_data = ((row[1:] / 255.0) * 0.99) + 0.01
        loss_val = nn.train(np.array(input_data, ndmin=2), np.array(target_data, ndmin=2))
        count += 1
        if count % check == 0:
            snapshots.append((count, nn.W2[:, :16].copy()))
            acc_hist.append((count, nn.accuracy(test_data[:1000])))
        if epochs == 1 and step % 10000 == 0:
            print(f'step {step:6d} : loss = {loss_val:.4f}')
    if epochs > 1:
        print(f'epoch {ep} 끝 : loss = {loss_val:.4f}')
print('학습 시간 =', datetime.now() - start_time)
print(f'테스트 정확도 = {nn.accuracy(test_data)*100:.2f} %')

> **[확인 3]** 오차역전파로 1 epoch 학습한 정확도는 ( 　　 ) % 이다. 원본 강의의 수치미분 버전(30,000장, 몇십 시간)의 정확도 94.34 % 와 비교해 보세요.

## 🎬 애니메이션 2. 은닉 노드는 무엇을 볼까?

은닉 노드 하나에 연결된 가중치 784 개 (W2 의 한 열) 를 28 × 28 그림으로 그리면 **그 노드가 좋아하는 무늬** 가 보입니다. (빨강 +, 파랑 −)
학습이 진행되며 무작위 점무늬가 **획(선)의 일부를 닮은 무늬** 로 바뀌는지 관찰하세요.

In [ ]:
#@title 🎬 애니메이션 2 실행 (실습 2 학습을 먼저 실행하세요)
fig = plt.figure(figsize=(12, 5.5))
gs = fig.add_gridspec(4, 8)
axs = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(16)]
axA = fig.add_subplot(gs[:, 4:])
frames = list(range(len(snapshots))) + [len(snapshots) - 1]*6
def draw(i):
    f = frames[i]
    cnt, Wk = snapshots[f]
    for j, ax in enumerate(axs):
        ax.clear(); w = Wk[:, j].reshape(IMG, IMG); m = np.abs(w).max()
        ax.imshow(w, cmap='bwr', vmin=-m, vmax=m); ax.axis('off')
    axA.clear()
    c = [a for a, _ in acc_hist[:f+1]]; v = [b*100 for _, b in acc_hist[:f+1]]
    axA.plot(c, v, 'o-', color='green'); axA.set_ylim(0, 100); axA.set_xlim(0, acc_hist[-1][0])
    axA.set_xlabel('학습한 사진 수'); axA.set_ylabel('정확도 (%)'); axA.grid(alpha=0.3)
    axA.yaxis.tick_right(); axA.yaxis.set_label_position('right')
    axA.set_title(f'{cnt}장 학습 → 정확도 {v[-1]:.1f} %', fontsize=13)
    fig.suptitle('은닉 노드 16개의 가중치 무늬', x=0.27, fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=250)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** 정확도는 처음 몇천 장 만에 ( 빠르게 / 천천히 ) 올라간다. 은닉 노드 무늬 중 숫자의 일부(동그라미, 세로 획 등)처럼 보이는 것이 있나요? (                     )

## ✏️ 연습 문제

> **[문제 1]** 은닉층을 2개 (784 - 100 - 50 - 10) 로 늘린다면 train 메서드에서 손실은 어떤 순서로 계산해야 할까요?
> loss_( 　 ) → loss_( 　 ) → loss_( 　 )

> **[문제 2]** 학습률을 0.3 대신 0.01 과 1.0 으로 바꿔 학습하고 정확도를 비교하세요. (아래 셀의 `______` 를 채우세요. 시간을 줄이려고 학습 데이터 10,000 장만 사용)

In [ ]:
for lr in [0.01, 0.3, ______]:
    np.random.seed(0)
    net = NeuralNetwork(IMG*IMG, 100, 10, lr)
    for row in training_data[:10000]:
        t = np.zeros(10) + 0.01; t[int(row[0])] = 0.99
        net.train(np.array(row[1:] / 255.0 * 0.99 + 0.01, ndmin=2), np.array(t, ndmin=2))
    print(f'학습률 {lr} : 정확도 = {net.accuracy(test_data)*100:.2f} %')

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
print('문제 1 : loss_4 (출력층) → loss_3 (은닉층2) → loss_2 (은닉층1)  : 출력층부터 거꾸로!')
print('문제 2 : 빈칸 1.0. 보통 0.01 은 너무 느리게 학습되어 정확도가 낮고, 0.3 이 가장 좋으며, 1.0 은 출렁여서 조금 떨어진다.')

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 은닉층 손실 | loss_2 = (loss_3 · W3<sup>T</sup>) × A2(1 − A2) |
| 일반 공식 | 현재 층 손실 = (다음 층 손실 · 다음 층 W<sup>T</sup>) × A(1 − A) |
| 갱신 | W ← W − α × (이전 층 출력<sup>T</sup> · 현재 층 손실),  b ← b − α × 현재 층 손실 |
| 순서 | 순전파(입력 → 출력) 후, 역전파(출력 → 입력) |
| 효과 | 수치미분보다 수만 배 빠르고, MNIST 1 epoch 로 약 95 % 정확도 |

### 🏆 도전 과제
지금 신경망은 사진의 784 픽셀을 **한 줄로 펴서** 넣기 때문에, 바로 옆에 붙어 있는 픽셀이라는 정보가 사라집니다. 사진의 **모양(공간 정보)** 을 살려서 학습하는 방법은? → 다음 시간: **CNN (합성곱 신경망)**

---

[⬅ 이전 : 제 12편 오차역전파 (1) - 연쇄법칙으로 한꺼번에 미분하기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/12_backprop_1.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/00_index.ipynb)　|　[다음 : 제 14편 CNN - 합성곱으로 사진의 특징 찾기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/14_cnn.ipynb)